In [16]:
# Simple benchmark comparing NumPy (CPU) vs CuPy (GPU) performance.

import time
from pathlib import Path
import numpy as np
import soundfile as sf
import pygaborstm as stm

# Load all 60 ripple files
data_dir = Path("data/mvripfft")
audio_files = sorted(data_dir.glob("*.wav"))
print(f"Found {len(audio_files)} audio files")

# Load and concatenate for a longer test signal
audios = []
for f in audio_files:
    audio, sr = sf.read(f)
    audios.append(audio)

audio = np.concatenate(audios)
print(f"Total audio: {len(audio)} samples ({len(audio)/sr:.1f}s at {sr}Hz)")

Found 60 audio files
Total audio: 2880000 samples (180.0s at 16000Hz)


In [17]:
# CPU Benchmark
config_cpu = stm.Config(use_gpu=False, resolution="low")
model_cpu = stm.PyGaborSTM(config_cpu)

# Warmup
_ = model_cpu.spectrogram(audios[0])

# Benchmark
start = time.perf_counter()
spec_cpu = model_cpu.spectrogram(audio)
rsf_cpu = model_cpu.rsf(spec_cpu)
cpu_time = time.perf_counter() - start

print(f"CPU time: {cpu_time:.3f}s")

CPU time: 169.165s


In [18]:
# GPU Benchmark
config_gpu = stm.Config(use_gpu=True, resolution="low")
model_gpu = stm.PyGaborSTM(config_gpu)

# Warmup (includes kernel compilation)
_ = model_gpu.spectrogram(audios[0])
_ = model_gpu.rsf(model_gpu.spectrogram(audios[0]))

# Benchmark
start = time.perf_counter()
spec_gpu = model_gpu.spectrogram(audio)
rsf_gpu = model_gpu.rsf(spec_gpu)
gpu_time = time.perf_counter() - start

print(f"GPU time: {gpu_time:.3f}s")

GPU time: 170.680s


In [19]:
# Results
print(f"\n{'='*40}")
print(f"Audio length: {len(audio)/sr:.1f}s")
print(f"Resolution:   low (60 filters)")
print(f"{'='*40}")
print(f"CPU time: {cpu_time:.3f}s")
print(f"GPU time: {gpu_time:.3f}s")
print(f"Speedup:  {cpu_time/gpu_time:.1f}x")
print(f"{'='*40}")

# Verify outputs match
spec_match = np.allclose(spec_cpu.data, spec_gpu.data, rtol=1e-4)
rsf_match = np.allclose(rsf_cpu.data, rsf_gpu.data, rtol=1e-4)
print(f"\nSpectrogram match: {spec_match}")
print(f"RSF match: {rsf_match}")


Audio length: 180.0s
Resolution:   low (60 filters)
CPU time: 169.165s
GPU time: 170.680s
Speedup:  1.0x

Spectrogram match: True
RSF match: True
